# TimeKAN: single-seed layer-depth study
21 runs: ETTh1, ETTh2, ETTm1 × L = 1, 2, 3, 4, 5, 10, 20. Seed 0 only.
All model training, evaluation and plotting run in **R**. Python only prepares Colab.
Read `docs/DEPTH_EXPERIMENT.md` for the fixed protocol and interpretation limits.
Select a GPU runtime. If Colab refuses a GPU, this notebook stops rather than silently
starting a long CPU study. You may explicitly change DEVICE to `cpu`.


In [ ]:
from pathlib import Path
import os, subprocess, shutil, sys
DEVICE = 'cuda'  # explicitly change to 'cpu' only if desired
if DEVICE == 'cuda':
    check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if check.returncode:
        raise RuntimeError('No GPU assigned. Connect a GPU runtime, or explicitly choose DEVICE=cpu.')
    print(check.stdout)
repo = Path('/content/atsa-timekan-depth')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/rviz190606/atsa-timekan.git',str(repo)],check=True)
os.chdir(repo)
print('Repository commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
assert (repo/'run_depth_experiment.R').exists(), 'The depth experiment must be pushed to main first.'


In [ ]:
# Install R and the official pre-built R torch package only when needed.
def stream(command):
    p = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line,end='',flush=True)
    if p.wait():
        raise RuntimeError('Command failed; inspect output above.')
if not shutil.which('Rscript'):
    stream(['apt-get','update','-qq'])
    stream(['apt-get','install','-y','-qq','r-base','r-base-dev','build-essential'])
check = subprocess.run(['Rscript','-e','library(torch); library(coro)'],capture_output=True,text=True)
if check.returncode:
    kind = 'cu128' if DEVICE == 'cuda' else 'cpu'
    install = f"""options(timeout=1800,repos=c(CRAN="https://cloud.r-project.org"))
v <- available.packages()["torch","Version"]
options(repos=c(torch=sprintf("https://torch-cdn.mlverse.org/packages/{kind}/%s/",v),CRAN="https://cloud.r-project.org"))
install.packages(c("torch","coro"),Ncpus=2)
library(torch); library(coro)
"""
    Path('/content/install_depth_torch.R').write_text(install)
    stream(['Rscript','/content/install_depth_torch.R'])
stream(['Rscript','run_depth_experiment.R','--task','self_test'])


In [ ]:
# Persistent backup. Reconnect to the same Drive folder to resume records.
from google.colab import drive
drive.mount('/content/drive')
backup = Path('/content/drive/MyDrive/TimeKAN_depth')
backup.mkdir(parents=True,exist_ok=True)
for folder in ['results/depth','figures/depth']:
    if (backup/folder).exists():
        shutil.copytree(backup/folder,repo/folder,dirs_exist_ok=True)
for filename in ['run_depth_experiment.R','timekan.R']:
    shutil.copy2(repo/filename,backup/filename)
print('Backup:',backup)


In [ ]:
# Only the depth sweep runs here; none of the old baseline suites are launched.
stream(['Rscript','run_depth_experiment.R','--device',DEVICE,'--backup',str(backup)])


In [ ]:
import pandas as pd, zipfile
from IPython.display import display, Image
table = pd.read_csv(repo/'results/depth/summary.csv')
assert len(table)==21 and not table.duplicated(['dataset','layers','seed']).any()
assert set(table.seed)=={0}
display(table[['dataset','layers','test_mse','test_mae','params','minutes','best_epoch','hit_epoch_limit']])
for filename in ['depth_accuracy_cost.png','depth_validation_curves.png']:
    display(Image(filename=str(repo/'figures/depth'/filename)))
bundle = backup/'TimeKAN_depth_results.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['results/depth','figures/depth']:
        for p in (repo/folder).rglob('*'):
            if p.is_file(): z.write(p,p.relative_to(repo))
    for f in ['run_depth_experiment.R','timekan.R','docs/DEPTH_EXPERIMENT.md']:
        z.write(repo/f,f)
print('DEPTH_SWEEP_VERIFIED: 21 runs. Saved',bundle)
